# Causality Dataset Creation

Using causality_dataset_creation_EDA.ipynb to create the final functions


Model 8:
1. Synapsida
2. Reptilia All

Model 9:
1. Synapsida
2. Reptilia Terres

Dataset format:
1. Genera name: one row per species
2. Lat range value for that species
3. Biome presence binary variables
4. ts: from combined combined_10_se_est.txts
5. te: from combined combined_10_se_est.txts
6. Age range: ts-te
7. Speciation rate: lam values from combined_10_per_species_rates.log. Remove 10% first iterations, calculate average per species
8. Extinction rate: mu values from combined_10_per_species_rates.log. Remove 10% first iterations, calculate average per species
9. Each climate variable (model 8 = isotopic, model 9 = BRIDGE), but per species by finding the climate variable's average value for that species' lifespan
    9.a. Nearest Neighbors (_nn) Method
    9.b. Weighted Interpolation (_wi) Method

## 1-8: Genera Data Function

Perform sections 1-8

In [177]:
import pandas as pd
import numpy as np

In [ ]:
# Need three input datasets: lats_file_final.csv, combined_10_se_est.txt and combined_10_per_species_rates.log

def genera_data(path_to_lats_file, path_to_combined_se_est, path_to_per_species_rates):
    ####################################################################################################
    #################### Load datasets
    lats_file = pd.read_csv(path_to_lats_file)
    se_est = pd.read_csv(path_to_combined_se_est, sep="\t")
    per_species_rates = pd.read_csv(path_to_per_species_rates, sep="\t")

    ####################################################################################################
    #################### SECTIONS 1-6
    # Needs to pass a check that "genus" column in lats file and "species" columns in se_est are identical in order to continue
    assert se_est['species'].equals(lats_file['genus']), "Genus columns do not match between se_est file and lats file."

    # ts and te need + 175 added back to them b/c my BDNN runs had 175 subtracted from all time values
    se_est['ts'] = se_est['ts'] + 175
    se_est['te'] = se_est['te'] + 175
    
    # time_span column from se_est
    se_est['time_span'] = se_est['ts'] - se_est['te']

    # Merge datasets on 'genus' column
    merged_data = pd.merge(lats_file, se_est[['species', 'ts', 'te', 'time_span']], left_on='genus', right_on='species', how='left')
    merged_data.drop(columns=['species'], inplace=True)

    # Need to pass check that no rows were lost
    assert len(merged_data) == len(lats_file), "Row count mismatch after merging se_est file and lats file"

    ####################################################################################################
    #################### SECTIONS 7-8
    per_species_rates = per_species_rates.drop(columns=per_species_rates.columns[-1])  # Drop last unnamed column, all per species rates files have this
    
    # Drop 10% burn-in (first 100 rows, inclusive)
    per_species_rates_burned = per_species_rates.iloc[100:, :]

    # assert that the # of rows = 900 (all combined logs were resampled to 100 so they should all have 1000 rows originally, - 10% burn-in)
    assert len(per_species_rates_burned) == 900, "Row count incorrect after dropping 10 percent burn-in from per species rates file"

    # New dataset which = the average value of eah column, maintaining the column heads
    per_species_rates_mean = per_species_rates_burned.mean().to_frame().T

    # Make sure that no columns were lost
    assert len(per_species_rates_mean.columns) == len(per_species_rates.columns), "Avg per species rates df column count mismatch with original per species rates df"

    # Separate _lam and _mu columns into their own dataframes
    per_species_rates_mean_lam = per_species_rates_mean.filter(like='_lam')
    per_species_rates_mean_mu = per_species_rates_mean.filter(like='_mu')

    # Make sure we didn't lose any columns. The + 1 is for the iterations column that we lost in the filtering 
    assert len(per_species_rates_mean_lam.columns) + len(per_species_rates_mean_mu.columns) + 1 == len(per_species_rates_mean.columns), "Column count mismatch between _lam and _mu cols dfs and original mean df"
    assert len(per_species_rates_mean_lam.columns) == len(per_species_rates_mean_mu.columns), "_lam and _mu dfs column count mismatch"

    # Transpose to have species as rows and averages as columns
    per_species_rates_mean_lam_T = per_species_rates_mean_lam.T.reset_index()
    per_species_rates_mean_lam_T.rename(columns={0: 'avg_lam', 'index': 'species'}, inplace=True)
    per_species_rates_mean_mu_T = per_species_rates_mean_mu.T.reset_index()
    per_species_rates_mean_mu_T.rename(columns={0: 'avg_mu', 'index': 'species'}, inplace=True)

    # Strip the suffixes from the species names to prepare for merging
    per_species_rates_mean_lam_T['species'] = per_species_rates_mean_lam_T['species'].str.replace('_lam', '')
    per_species_rates_mean_mu_T['species'] = per_species_rates_mean_mu_T['species'].str.replace('_mu', '')

    # Length check
    assert per_species_rates_mean_lam_T.shape[0] == per_species_rates_mean_mu_T.shape[0] == merged_data.shape[0], "_lam, _mu, merged dfs row count mismatch"

    # Merge with main df
    merged_data = pd.merge(merged_data, per_species_rates_mean_lam_T, left_on='genus', right_on='species', how='left')
    merged_data = pd.merge(merged_data, per_species_rates_mean_mu_T, left_on='genus', right_on='species', how='left')
    merged_data.drop(columns=['species_x', 'species_y'], inplace=True)

    return merged_data


## Climate Data

Perform section 9 in TWO ways:
1. Nearest Neighbor
2. Weighted Interpolation

See the causality_dataset_creation_EDA.ipynb file to see discussion of the pros/cons of each method

In [ ]:
def nearest_neighbor(path_to_climate_data, merged_df):
    ####################################################################################################
    #################### Load climate dataset
    climate_data = pd.read_csv(path_to_climate_data)

    ####################################################################################################
    #################### SECTION 9
    original_row_count = len(merged_df)
    merged_df_nn = merged_df.copy()

    # Warning trackers if extrapolation or distant assignments occur (if a genera l's lifespan is outside climate data range or >1 Myr from midpoint, repsectively)
    extrapolated_genera = []
    distant_assignment_genera = []  # >1 Myr from midpoint
    climate_time_min = climate_data['Time'].min()
    climate_time_max = climate_data['Time'].max()
    
    # Automatically detect climate columns (all columns except 'Time')
    climate_cols = [col for col in climate_data.columns if col != 'Time']
    
    # Initialize new columns with _NN suffix
    for col in climate_cols:
        merged_df_nn[f'{col}_NN'] = np.nan
    
    for index, row in merged_df_nn.iterrows():
        ts = row['ts']
        te = row['te']
        
        # Filter climate data for the time span of the genus
        climate_filtered = climate_data[(climate_data['Time'] <= ts) & (climate_data['Time'] >= te)]
        
        # Calculate mean values for all climate columns
        mean_values = climate_filtered[climate_cols].mean() # Is this matematically poor if there are missing rows in isotopic data? Some myrs have no data
        
        for col in climate_cols:
            merged_df_nn.at[index, f'{col}_NN'] = mean_values[col]
        
        # If any null, assign values from closest time point to midpoint of lifespan
        if any(pd.isna(merged_df_nn.at[index, f'{col}_NN']) for col in climate_cols):
            midpoint = (ts + te) / 2
            closest_idx = (climate_data['Time'] - midpoint).abs().idxmin()
            closest_time = climate_data.loc[closest_idx]

            # Track if this is extrapolation (genus lifespan entirely outside climate data range)
            if ts < climate_time_min or te > climate_time_max:
                extrapolated_genera.append(row['genus'])
            
            # Track if nearest neighbor is >1 Myr away (very distant assignment)
            distance = abs(closest_time['Time'] - midpoint)
            if distance > 1.0:
                distant_assignment_genera.append((row['genus'], distance))
            
            for col in climate_cols:
                # Only replace if this specific column is null
                if pd.isna(merged_df_nn.at[index, f'{col}_NN']):
                    merged_df_nn.at[index, f'{col}_NN'] = closest_time[col]    

    assert len(merged_df_nn) == original_row_count, "The original dataframe and the nearest neighbor merged dataframe have different row counts."
    
    # Print warnings about extrapolation or distant assignments
    if extrapolated_genera:
        print(f"Warning: {len(extrapolated_genera)} genera had lifespans entirely outside climate data range "
              f"({climate_time_min}-{climate_time_max} Ma) and were assigned values via extrapolation: "
              f"{extrapolated_genera[:5]}{'...' if len(extrapolated_genera) > 5 else ''}")
    
    if distant_assignment_genera:
        distant_assignment_genera.sort(key=lambda x: x[1], reverse=True)
        print(f"Warning: {len(distant_assignment_genera)} genera were assigned climate values from time points >1 Myr from their midpoint. Most extreme cases: "
              f"{distant_assignment_genera[:5]}{'...' if len(distant_assignment_genera) > 5 else ''}")
        
    return merged_df_nn

In [ ]:
def weighted_interpolation(path_to_climate_data, merged_df):
    ####################################################################################################
    #################### Load climate dataset
    climate_data = pd.read_csv(path_to_climate_data)

    ####################################################################################################
    #################### SECTION 10
    original_row_count = len(merged_df)
    merged_df_wi = merged_df.copy()

    # Warning trackers if extrapolation or distant assignments occur (if a genera l's lifespan is outside climate data range or >1 Myr from midpoint, repsectively)
    extrapolated_genera = []
    climate_time_min = climate_data['Time'].min()
    climate_time_max = climate_data['Time'].max()
    
    # Automatically detect climate columns (all columns except 'Time')
    climate_cols = [col for col in climate_data.columns if col != 'Time']
    
    # Initialize new columns with _WI suffix
    for col in climate_cols:
        merged_df_wi[f'{col}_WI'] = np.nan
    
    # Get sorted list of available time points for efficient lookup
    available_times = sorted(climate_data['Time'].unique(), reverse=True)
    
    for index, row in merged_df_wi.iterrows():
        ts = row['ts']  # Time of speciation (older)
        te = row['te']  # Time of extinction (younger)
        midpoint = (ts + te) / 2
        
        # Filter climate data for the time span of the genus
        climate_filtered = climate_data[(climate_data['Time'] <= ts) & (climate_data['Time'] >= te)]
        
        if len(climate_filtered) > 0:
            # Direct overlap exists - use arithmetic mean
            mean_values = climate_filtered[climate_cols].mean()
            for col in climate_cols:
                merged_df_wi.at[index, f'{col}_WI'] = mean_values[col]
        
        else:
            # No direct overlap - use weighted interpolation
            
            # Find t_upper: smallest time point >= ts (closest point older than or at speciation)
            upper_candidates = [t for t in available_times if t >= ts]
            t_upper = min(upper_candidates) if upper_candidates else available_times[0]
            
            # Find t_lower: largest time point <= te (closest point younger than or at extinction)
            lower_candidates = [t for t in available_times if t <= te]
            t_lower = max(lower_candidates) if lower_candidates else available_times[-1]

            # Track if this required extrapolation beyond climate data bounds
            if not upper_candidates or not lower_candidates:
                extrapolated_genera.append(row['genus'])
            
            # Calculate weights based on inverse distance to midpoint
            if t_upper != t_lower:
                w_lower = (t_upper - midpoint) / (t_upper - t_lower)
                w_upper = (midpoint - t_lower) / (t_upper - t_lower)
            else:
                # Edge case: both brackets are the same point (extrapolation)
                w_lower = 1.0
                w_upper = 0.0
            
            # Get climate values at the bracketing points
            lower_row = climate_data[climate_data['Time'] == t_lower].iloc[0]
            upper_row = climate_data[climate_data['Time'] == t_upper].iloc[0]
            
            # Calculate weighted averages for all climate columns
            for col in climate_cols:
                weighted_val = lower_row[col] * w_lower + upper_row[col] * w_upper
                merged_df_wi.at[index, f'{col}_WI'] = weighted_val
    
    assert len(merged_df_wi) == original_row_count, "The original dataframe and the weighted interpolation merged dataframe have different row counts."
    
    # Print warning
    if extrapolated_genera:
        print(f"Warning: {len(extrapolated_genera)} genera had lifespans that required extrapolation beyond "
              f"climate data range ({climate_time_min}-{climate_time_max} Ma). These genera were assigned values from the nearest available boundary: "
              f"{extrapolated_genera[:5]}{'...' if len(extrapolated_genera) > 5 else ''}")
        
    return merged_df_wi

## Old, isotopic-only functions

In [ ]:
# def nearest_neighbor(path_to_climate_data, merged_df):
#     ####################################################################################################
#     #################### Load climate dataset
#     climate_data = pd.read_csv(path_to_climate_data)

#     ####################################################################################################
#     #################### SECTION 9
#     original_row_count = len(merged_df)
#     merged_df_nn = merged_df.copy()
    
#     merged_df_nn['mean_pt_1myr_z_trans_NN'] = None
#     merged_df_nn['Mod_R_deltaTMyr_pt_1myr_z_trans_NN'] = None
    
#     for index, row in merged_df_nn.iterrows():
#         ts = row['ts']
#         te = row['te']
        
#         # Filter isotopic data for the time span of the genus
#         isotopic_filtered = climate_data[(climate_data['Time'] <= ts) & (climate_data['Time'] >= te)]
        
#         # Calculate mean values for the relevant columns
#         mean_values = isotopic_filtered[['mean_pt_1myr_z_trans', 'Mod_R_deltaTMyr_pt_1myr_z_trans']].mean()
        
#         merged_df_nn.at[index, 'mean_pt_1myr_z_trans_NN'] = mean_values['mean_pt_1myr_z_trans']
#         merged_df_nn.at[index, 'Mod_R_deltaTMyr_pt_1myr_z_trans_NN'] = mean_values['Mod_R_deltaTMyr_pt_1myr_z_trans']
        
#         # If null, assign value from closest time point to midpoint of lifespan
#         if pd.isna(merged_df_nn.at[index, 'mean_pt_1myr_z_trans_NN']) or pd.isna(merged_df_nn.at[index, 'Mod_R_deltaTMyr_pt_1myr_z_trans_NN']):
#             midpoint = (ts + te) / 2
#             closest_idx = (climate_data['Time'] - midpoint).abs().idxmin()
#             closest_time = climate_data.loc[closest_idx]
            
#             merged_df_nn.at[index, 'mean_pt_1myr_z_trans_NN'] = closest_time['mean_pt_1myr_z_trans']
#             merged_df_nn.at[index, 'Mod_R_deltaTMyr_pt_1myr_z_trans_NN'] = closest_time['Mod_R_deltaTMyr_pt_1myr_z_trans']
    
#     assert len(merged_df_nn) == original_row_count, "The original dataframe and the nearest neighbor merged dataframe have different row counts."
    
#     return merged_df_nn

In [ ]:
# def weighted_interpolation(path_to_climate_data, merged_df):
#     ####################################################################################################
#     #################### Load climate dataset
#     climate_data = pd.read_csv(path_to_climate_data)

#     ####################################################################################################
#     #################### SECTION 10
#     original_row_count = len(merged_df)
#     merged_df_wi = merged_df.copy()
    
#     merged_df_wi['mean_pt_1myr_z_trans_WI'] = None
#     merged_df_wi['Mod_R_deltaTMyr_pt_1myr_z_trans_WI'] = None
    
#     # Get sorted list of available time points for efficient lookup
#     available_times = sorted(climate_data['Time'].unique(), reverse=True)
    
#     for index, row in merged_df_wi.iterrows():
#         ts = row['ts']  # Time of speciation (older)
#         te = row['te']  # Time of extinction (younger)
#         midpoint = (ts + te) / 2
        
#         # Filter climate data for the time span of the genus
#         climate_filtered = climate_data[(climate_data['Time'] <= ts) & (climate_data['Time'] >= te)]
        
#         if len(climate_filtered) > 0:
#             # Direct overlap exists - use arithmetic mean
#             mean_pt = climate_filtered['mean_pt_1myr_z_trans'].mean()
#             mod_r = climate_filtered['Mod_R_deltaTMyr_pt_1myr_z_trans'].mean()
        
#         else:
#             # No direct overlap - use weighted interpolation
            
#             # Find t_upper: smallest time point >= ts (closest point older than or at speciation)
#             upper_candidates = [t for t in available_times if t >= ts]
#             t_upper = min(upper_candidates) if upper_candidates else available_times[0]
            
#             # Find t_lower: largest time point <= te (closest point younger than or at extinction)
#             lower_candidates = [t for t in available_times if t <= te]
#             t_lower = max(lower_candidates) if lower_candidates else available_times[-1]
            
#             # Calculate weights based on inverse distance to midpoint
#             if t_upper != t_lower:
#                 w_lower = (t_upper - midpoint) / (t_upper - t_lower)
#                 w_upper = (midpoint - t_lower) / (t_upper - t_lower)
#             else:
#                 # Edge case: both brackets are the same point (extrapolation)
#                 w_lower = 1.0
#                 w_upper = 0.0
            
#             # Get climate values at the bracketing points
#             lower_row = climate_data[climate_data['Time'] == t_lower].iloc[0]
#             upper_row = climate_data[climate_data['Time'] == t_upper].iloc[0]
            
#             # Calculate weighted averages
#             mean_pt = lower_row['mean_pt_1myr_z_trans'] * w_lower + upper_row['mean_pt_1myr_z_trans'] * w_upper
#             mod_r = lower_row['Mod_R_deltaTMyr_pt_1myr_z_trans'] * w_lower + upper_row['Mod_R_deltaTMyr_pt_1myr_z_trans'] * w_upper
        
#         merged_df_wi.at[index, 'mean_pt_1myr_z_trans_WI'] = mean_pt
#         merged_df_wi.at[index, 'Mod_R_deltaTMyr_pt_1myr_z_trans_WI'] = mod_r
    
#     assert len(merged_df_wi) == original_row_count, "The original dataframe and the weighted interpolation merged dataframe have different row counts."
    
#     return merged_df_wi